In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter

# Load your sentence dataset
df = pd.read_csv("recombined_corpsfinalSentence3_SOAP.csv")  # adjust path if needed

# 1) 80% train vs 20% holdout — grouped by encounter_id
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_hold = next(gss1.split(df, groups=df["encounter_id"]))
train = df.iloc[idx_train].copy()
hold  = df.iloc[idx_hold].copy()

# 2) Split the 20% holdout into 10% dev and 10% test — still grouped by encounter_id
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
idx_dev, idx_test = next(gss2.split(hold, groups=hold["encounter_id"]))
dev  = hold.iloc[idx_dev].copy()
test = hold.iloc[idx_test].copy()

# Sanity checks — no encounter leakage
assert set(train["encounter_id"]).isdisjoint(dev["encounter_id"])
assert set(train["encounter_id"]).isdisjoint(test["encounter_id"])
assert set(dev["encounter_id"]).isdisjoint(test["encounter_id"])

# Optional: check class coverage
def coverage(df_part):
    return Counter(df_part["class"])

print("Rows:", len(train), len(dev), len(test))
print("Encounters:", train["encounter_id"].nunique(), dev["encounter_id"].nunique(), test["encounter_id"].nunique())
print("Missing classes in train:", [c for c in sorted(df["class"].unique()) if coverage(train).get(c,0)==0])

# Save splits
train.to_csv("sentences_train_SOAP.csv", index=False)
dev.to_csv("sentences_val_SOAP.csv", index=False)
test.to_csv("sentences_test_SOAP.csv", index=False)


Rows: 5069 656 659
Encounters: 165 21 21
Missing classes in train: []
